# Cloud Provider Analytics — Pipeline completo

```text
Landing → Bronze → Silver → Gold → Serving (AstraDB)
```

Orquestación end-to-end. La lógica vive en `src/jobs/`; este notebook ejecuta cada capa.

**Prerrequisitos:** `datalake/landing/`, keyspace `cloud_analytics` en Astra, token + bundle.

In [ ]:
# Setup (Colab o local) + Spark
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    !pip install -q pyspark cassandra-driver
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    PROJECT_ROOT = Path("/content/drive/MyDrive/cloud-provider-analytics")
else:
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from IPython.display import display
from pyspark.sql import SparkSession

from src.cassandra.client import is_astra_configured
from src.config import BRONZE, CASSANDRA_KEYSPACE, DATA_ROOT, GOLD, LANDING, SILVER
from src.jobs.bronze_streaming import (
    USAGE_EVENTS_BRONZE_PATH,
    USAGE_EVENTS_CHECKPOINT_PATH,
    USAGE_EVENTS_LANDING_GLOB,
)
from src.jobs.gold import ORG_DAILY_USAGE_BY_SERVICE
from src.jobs.silver import USAGE_EVENTS_SILVER
from src.schemas.bronze_streaming import WATERMARK_DELAY
from src.spark.performance import configure_spark_performance

spark = (
    SparkSession.builder.appName("cloud-provider-analytics")
    .master("local[*]")
    .getOrCreate()
)
configure_spark_performance(spark)
spark.sparkContext.setLogLevel("WARN")

print(f"PROJECT_ROOT:  {PROJECT_ROOT}")
print(f"DATA_ROOT:     {DATA_ROOT}")
print(f"Spark:         {spark.version}")
print(f"Astra:         {is_astra_configured()}")
print(f"KEYSPACE:      {CASSANDRA_KEYSPACE}")


## 1. Batch Bronze — maestros CSV

Ingesta de los 7 maestros del landing → Parquet con `ingest_ts`, `source_file` y dedupe.

In [ ]:
from src.jobs.bronze_batch import run_batch_bronze, validate_bronze_uniqueness

batch_results = run_batch_bronze(spark)
display(pd.DataFrame(batch_results)[
    ["dataset_name", "raw_count", "deduped_count", "removed_duplicates", "written_count"]
])
display(pd.DataFrame(validate_bronze_uniqueness(spark)))


## 2. Streaming Bronze — usage_events

Structured Streaming desde JSONL, watermark, dedupe `event_id`, late data y reparquet.

> **Corrida limpia (parcial):** `reset_state=True` regenera Bronze desde landing con normalización temprana.
> Replay: `ingest_ts = event_ts` (watermark 60d). Producción: `ingest_ts = now()`.

In [ ]:
from src.jobs.bronze_streaming import run_streaming_bronze, validate_bronze_streaming

print(f"Landing: {USAGE_EVENTS_LANDING_GLOB}")
streaming_result = run_streaming_bronze(spark, reset_state=True)
display(pd.DataFrame([{k: v for k, v in streaming_result.items() if k != "reparquet"}]))
display(pd.DataFrame([validate_bronze_streaming(spark)]))


## 3. Silver

Maestros + `usage_events` (grano evento), `org_service_daily`, anomalías de costo y quarantine.

In [ ]:
from src.jobs.silver import run_silver, validate_silver

silver_results = run_silver(spark)
display(pd.DataFrame(silver_results))

events = next(r for r in silver_results if r["dataset_name"] == "usage_events")
display(pd.DataFrame([{
    "bronze": events["raw_count"],
    "silver_valid": events["valid_count"],
    "quarantine": events["quarantine_count"],
    "late_quarantined": events["late_arrivals_quarantined"],
    "negative_cost_quarantined": events.get("negative_cost_quarantined"),
}]))
if events.get("quarantine_sample"):
    print("Muestra quarantine (usage_events):")
    display(pd.DataFrame(events["quarantine_sample"]))

display(pd.DataFrame([validate_silver(spark)]))



## 4. Gold — marts de negocio

**Astra (5 tablas):** FinOps diario, top servicios (rolling 14d), tickets, revenue, GenAI.

**Solo Parquet:** `cost_anomaly_mart`, `nps_by_org_date`, `marketing_touches_by_org_channel`.

In [ ]:
from src.jobs.gold import run_gold, validate_gold

display(pd.DataFrame(run_gold(spark)))
display(pd.DataFrame([validate_gold(spark)]))


## 5. Serving — carga Gold → Astra

Carga vía `foreachBatch` (consigna §5). Consultas CQL en §7.

In [ ]:
from src.jobs.serving_cassandra import run_serving

if not is_astra_configured():
    print("Omitido: configurar ASTRA_DB_APPLICATION_TOKEN y ASTRA_DB_SECURE_BUNDLE_PATH.")
else:
    load = run_serving(spark).get("load")
    if load:
        display(pd.DataFrame(load))


## 6. Idempotencia

Re-ejecución batch Bronze → streaming (sin `reset_state`) → Silver → Gold. Criterio: conteos estables, `event_id` únicos, balance Silver.

In [ ]:
from src.jobs.bronze_batch import run_batch_bronze
from src.jobs.bronze_streaming import run_streaming_bronze
from src.jobs.gold import (
    GENAI_TOKENS_BY_ORG_DATE,
    ORG_DAILY_USAGE_BY_SERVICE,
    REVENUE_BY_ORG_MONTH,
    TICKETS_BY_ORG_DATE,
    run_gold,
)
from src.jobs.silver import USAGE_EVENTS_QUARANTINE, run_silver

MASTER_DATASETS = [
    "customers_orgs", "users", "billing_monthly", "resources",
    "support_tickets", "marketing_touches", "nps_surveys",
]


def lake_snapshot() -> dict[str, int]:
    bronze_events = spark.read.parquet(USAGE_EVENTS_BRONZE_PATH)
    silver_valid = spark.read.parquet(USAGE_EVENTS_SILVER)
    quarantine = 0
    if os.path.isdir(USAGE_EVENTS_QUARANTINE):
        quarantine = spark.read.parquet(USAGE_EVENTS_QUARANTINE).count()
    return {
        "bronze_masters": sum(spark.read.parquet(f"{BRONZE}/{d}").count() for d in MASTER_DATASETS),
        "bronze_events": bronze_events.count(),
        "bronze_events_distinct": bronze_events.select("event_id").distinct().count(),
        "silver_valid": silver_valid.count(),
        "silver_quarantine": quarantine,
        "gold_finops": spark.read.parquet(ORG_DAILY_USAGE_BY_SERVICE).count(),
        "gold_revenue": spark.read.parquet(REVENUE_BY_ORG_MONTH).count(),
        "gold_tickets": spark.read.parquet(TICKETS_BY_ORG_DATE).count(),
        "gold_genai": spark.read.parquet(GENAI_TOKENS_BY_ORG_DATE).count(),
    }


before = lake_snapshot()
run_batch_bronze(spark)
streaming_rerun = run_streaming_bronze(spark, reset_state=False)
run_silver(spark)
run_gold(spark)
after = lake_snapshot()

comparison = pd.DataFrame(
    [{"metric": k, "before": before[k], "after": after[k], "ok": before[k] == after[k]}
     for k in before]
)
display(comparison)

lake_ok = (
    comparison["ok"].all()
    and after["bronze_events"] == after["bronze_events_distinct"]
    and after["bronze_events"] == after["silver_valid"] + after["silver_quarantine"]
)
print(f"Lake idempotencia: {'OK' if lake_ok else 'FAIL'} | streaming written={streaming_rerun['written_count']}")

if is_astra_configured():
    from src.cassandra.client import get_cassandra_session
    from src.cassandra.schema import TABLE_ORG_DAILY
    from src.jobs.serving_cassandra import load_org_daily_usage_by_service

    sample_org = (
        spark.read.parquet(ORG_DAILY_USAGE_BY_SERVICE)
        .select("org_id")
        .limit(1)
        .collect()[0]["org_id"]
    )
    session, cluster = get_cassandra_session()
    try:
        before_c = session.execute(
            f"SELECT COUNT(*) FROM {TABLE_ORG_DAILY} WHERE org_id = %s",
            (sample_org,),
        ).one()[0]
        load_org_daily_usage_by_service(spark, session)
        after_c = session.execute(
            f"SELECT COUNT(*) FROM {TABLE_ORG_DAILY} WHERE org_id = %s",
            (sample_org,),
        ).one()[0]
        cassandra_ok = before_c == after_c
        display(pd.DataFrame([{
            "sample_org_id": sample_org,
            "rows_before": before_c,
            "rows_after": after_c,
            "ok": cassandra_ok,
        }]))
        print(f"Cassandra idempotencia (sample org): {'OK' if cassandra_ok else 'FAIL'}")
    finally:
        cluster.shutdown()
else:
    print("Cassandra idempotencia: omitida (Astra no configurado).")


## 7. Consultas AstraDB (#1–#5)

Una celda por consulta: **CQL literal** impreso y ejecutado (`src.cassandra.selects`, alineado a `cql/01–05`).

Prerrequisito: §5 o tablas ya cargadas en `cloud_analytics`.

In [ ]:
from src.cassandra.demo import close_demo_session, open_demo_session
from src.cassandra.schema import TICKETS_CRITICAL_LOOKBACK_DAYS, TOP_SERVICES_LOOKBACK_DAYS

demo = open_demo_session(spark)

if demo is None:
    print("Astra no configurado.")
else:
    p = demo.params
    print(f"DEMO_ORG_ID = {p.org_id}")
    print(f"#2 ventana = {p.period_start} → {p.period_end} ({TOP_SERVICES_LOOKBACK_DAYS}d)")
    print(f"#3 ventana = {p.q3_start} → {p.q3_end} ({TICKETS_CRITICAL_LOOKBACK_DAYS}d, severity={p.q3_severity})")


In [ ]:
# Consulta #1 — costos y requests diarios por servicio
if demo is None:
    print("Consulta #1 omitida.")
else:
    display_cql_select(
        demo.session,
        daily_costs_and_requests(demo.params.org_id, demo.params.q1_start, demo.params.q1_end),
    )



In [ ]:
# Consulta #2 — top-N servicios por costo (ventana rolling 14 días)
if demo is None:
    print("Consulta #2 omitida.")
else:
    display_cql_select(
        demo.session,
        top_services_by_cost(
            demo.params.org_id, demo.params.period_end, demo.params.top_n,
        ),
    )



In [ ]:
# Consulta #3 — tickets críticos (severity=high) y SLA
if demo is None:
    print("Consulta #3 omitida.")
else:
    display_cql_select(
        demo.session,
        critical_tickets_sla(
            demo.params.org_id,
            demo.params.q3_severity,
            demo.params.q3_start,
            demo.params.q3_end,
        ),
    )



In [ ]:
# Consulta #4 — revenue mensual USD
if demo is None:
    print("Consulta #4 omitida.")
else:
    display_cql_select(
        demo.session,
        monthly_revenue(demo.params.org_id, demo.params.q4_start, demo.params.q4_end),
    )



In [ ]:
# Consulta #5 — tokens GenAI y costo estimado por día
if demo is None:
    print("Consulta #5 omitida.")
else:
    display_cql_select(
        demo.session,
        genai_tokens_daily(demo.params.org_id, demo.params.q5_start, demo.params.q5_end),
    )

close_demo_session(demo)

